# MLflow Models: Custom pyfunc model

This notebook is divided into small cells with markdown explanation and line-by-line comments. It uses `datasets/customer_churn_500.csv` from this folder.

## 1. Import required libraries

This cell imports MLflow, pandas, NumPy, matplotlib, and scikit-learn utilities.

In [ ]:
# Import os to create local folders and manage paths.
import os

# Import warnings to hide non-critical warning messages during training.
import warnings

# Ignore warnings for cleaner notebook output.
warnings.filterwarnings("ignore")

# Import MLflow for experiment tracking and model management.
import mlflow

# Import MLflow sklearn integration for logging sklearn models.
import mlflow.sklearn

# Import pandas for reading and processing CSV data.
import pandas as pd

# Import numpy for numerical calculations.
import numpy as np

# Import matplotlib for creating charts and artifact images.
import matplotlib.pyplot as plt

# Import train_test_split for splitting data into train and test sets.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for separate numerical and categorical processing.
from sklearn.compose import ColumnTransformer

# Import encoders and scalers for preprocessing.
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Import Pipeline to combine preprocessing and model into one object.
from sklearn.pipeline import Pipeline

# Import ML models.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Import classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Import regression metrics.
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 2. Configure MLflow

This cell configures MLflow to store runs locally in the current folder under `mlruns`.

In [ ]:
# Create local MLflow tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Store tracking data locally in this folder.
mlflow.set_tracking_uri("file:./mlruns")

# Set experiment name for this notebook.
mlflow.set_experiment("03_mlflow_models_05_custom_pyfunc_model")

# Define the dataset path used by this notebook.
DATA_PATH = "datasets/customer_churn_500.csv"

# Print confirmation.
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Dataset path:", DATA_PATH)

## 3. Load and inspect dataset

The dataset contains 500 realtime-style customer records with categorical and numerical columns.

In [ ]:
# Read the CSV dataset from the local datasets folder.
df = pd.read_csv(DATA_PATH)

# Display the first five records.
display(df.head())

# Display dataset shape.
print("Dataset shape:", df.shape)

# Display column names.
print("Columns:", df.columns.tolist())

## 4. Prepare features and target

For classification, we predict `churn`. For regression, we predict `monthly_charges`.

In [ ]:
# Define classification target column.
target_column = "churn"

# Remove customer ID and target column from features.
X = df.drop(columns=["customer_id", target_column])

# Store the classification target.
y = df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Print feature summary.
print("Target column:", target_column)
print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)

## 5. Build preprocessing pipeline

Categorical columns are one-hot encoded. Numerical columns are scaled.

In [ ]:
# Create preprocessing logic for numerical and categorical columns.
preprocessor = ColumnTransformer(
    transformers=[
        # Scale numerical columns.
        ("num", StandardScaler(), numerical_columns),

        # Convert categorical columns into numeric one-hot encoded columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Print the preprocessing object.
preprocessor

## 6. Split the dataset

Training data is used to train the model. Testing data is used to evaluate the model.

In [ ]:
# Split data into training and testing datasets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print split sizes.
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 7. Create custom pyfunc model

This example packages business rules as an MLflow model.

In [ ]:
# Import MLflow pyfunc module.
import mlflow.pyfunc

# Define custom model class.
class CustomerRiskRuleModel(mlflow.pyfunc.PythonModel):

    # Define prediction logic.
    def predict(self, context, model_input):

        # Copy input data.
        data = model_input.copy()

        # Find customers with many support tickets.
        high_ticket_risk = data["support_tickets"] >= 4

        # Find month-to-month contract customers.
        month_to_month_risk = data["contract_type"] == "Month-to-month"

        # Find low-tenure customers.
        low_tenure_risk = data["tenure_months"] < 12

        # Combine rules.
        risk = high_ticket_risk | (month_to_month_risk & low_tenure_risk)

        # Return risk labels.
        return pd.Series(np.where(risk, "high_churn_risk", "low_churn_risk"))

## 8. Log and load custom model

The model is saved with MLflow and loaded back for prediction.

In [ ]:
# Create input example.
input_example = df.drop(columns=["customer_id", "churn"]).head(5)

# Start MLflow run.
with mlflow.start_run(run_name="custom_customer_risk_pyfunc"):

    # Log custom pyfunc model.
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=CustomerRiskRuleModel(),
        input_example=input_example
    )

    # Create model URI.
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"

# Load model.
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Predict on sample data.
predictions = loaded_model.predict(input_example)

# Display predictions.
print("Model URI:", model_uri)
print(predictions)